# Preparation

In [106]:
import json
import pandas as pd
from collections import Counter
import requests
import re
from pathlib import Path

In [66]:
home = Path.home()

# Functions

In [42]:
TOKEN_RE = re.compile(r"\d+|[^\W\d_]+|[.,/:;()\[\]-]")

def skeleton(text):
    out = []
    prev = None

    for tok in TOKEN_RE.findall(text):
        if tok[0].isdigit():
            kind = "N"
        elif tok[0].isalpha():
            kind = "W"
        else:
            kind = tok

        # Collapse consecutive words
        if kind == "W" and prev == "W":
            continue

        out.append(kind)
        prev = kind

    return "".join(out)

In [57]:
def walk_keys(obj, prefix=""):
    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key
            yield path
            yield from walk_keys(value, path)

# Analysis

## Initial data

In [67]:
structure_counter = Counter()


objects = []

with open(f"{home}/code/data/shbd/shb.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]

skeleton_notes = []

example = {}
	
print(len(objects))

for object in objects:
	entity = object["@graph"][1]
	if "hasNote" in entity:
		note = entity["hasNote"][0]["label"]
		pattern = skeleton(note)
		skeleton_notes.append(pattern)
		structure_counter.update([pattern])

		example.setdefault(pattern, note)

print(len(skeleton_notes))
print(*skeleton_notes[:3], sep="\n")

79114
79024
W,W,W-W:WN:(W).-W:W,WN-N,N,N,W.N-N.-W.W-W,WN-N
W,W.W,W.-W:W,WN-N,N,N:N,W.N-N
W,W,W:W.-W:W,WN-N,N:N,W.N-N


### Count properties

In [61]:
property_counts = Counter()
subject_counts = Counter()

for object in objects:
	entity = object["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
    
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )

In [62]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

79114  @id
79114  @type
79114  category
79114  publication
79114  isPartOf
79114  part
79114  associatedMedia
79114  marc:primaryProvisionActivity
79114  marc:primaryProvisionActivity.year
79114  marc:primaryProvisionActivity.@type
79114  instanceOf
79114  instanceOf.@id
79024  hasNote


### Inspect structure of descriptions

In [53]:
print("| Count | Pattern | Example |")
print("|------:|---------|---------|")

for pattern, count in structure_counter.most_common(20):
    ex = example[pattern].replace("|", "\\|")  # Escape pipes if any
    print(f"| {count} | `{pattern}` | {ex} |")

| Count | Pattern | Example |
|------:|---------|---------|
| 842 | `W,W.,W.(WN,W.N-N.)` | Dalgren, L., Ur den nyaste tyska Arndtlitteraturen. (HT 1922, s. 247-249.) |
| 569 | `W,W.,W.(W.N(N),W.N-N.)` | Brulin, H., Das schwedische Archivwesen. (Archivalische Zeitschr. 38 (1929),s. 151-177.) |
| 566 | `W,W,W.(WN,W.N-N.)` | Lundberg, Erik, Nyare forskning över svensk byggnadshistoria. (Rig 1932,s. 105-127.) |
| 504 | `W,W,W.-WN.NN.` | Jagerskiold, Stig, Svea hovrätt jubilerar. - SvD 17.2 1964. |
| 417 | `W,W.,W.(WN(N),W.N-N.)` | Berghman, A., Heraldisk litteratur. (MRÄ 4 (1935), s. 9-38.) |
| 414 | `W,W,W.(W.N(N),W.N-N.)` | Floderus, Erik, Våra äldsta mynt. (Kooperatören. 17 (1930), s. 120-126.) |
| 321 | `W,W,W.(WN(N),W.N-N.)` | Söderberg, Bengt, Gotländska glasmålningar med länsherrevapen. (GA 5(1933), s. 37-44.) |
| 294 | `W,W,W.-WN(N),W.N-N.` | Åkerman, Sune, Projects and research priorities. - Historisk tidskrift 90 (1970),  s. 47-67. |
| 266 | `W,W,W.(WN/NN.)` | Leide, Arvid, Danie

## Enriched data

In [211]:
objects = []

with open(f"{home}/code/data/shbd/shb-cleaned-with-subjects.jsonld.lines", "r", encoding="utf-8") as f:
	objects = [json.loads(row) for row in f]
	
print(len(objects))


79114


In [212]:
property_counts = Counter()
subject_counts = Counter()

instances = []
for object in objects:
	entity = object["@graph"]["@graph"][1]
	property_counts.update(walk_keys(entity))
      
	subjects = entity.get("instanceOf", {}).get("subject", [])
	subject_counts.update(
          subject["@id"]
              for subject in subjects
                    )
	
	instances.append(entity)

print(instances[:3])

[{'@id': 'https://libris-qa.kb.se/dataset/shb/1#it', '@type': 'PhysicalResource', 'category': [{'@id': 'https://id.kb.se/term/saobf/ComponentPart'}, {'@id': 'https://id.kb.se/term/saobf/Print'}], 'instanceOf': {'@type': 'Monograph', 'category': [{'@id': 'https://id.kb.se/term/rda/Text'}]}, 'hasTitle': {'@type': 'Title', 'mainTitle': 'Malmö-litteratur', 'subtitle': 'bibliografiska noteringar för år1975 : (med tillägg från föregående år).'}, 'responsibilityStatement': 'Andersson, Per', 'partOf': [{'@type': 'Instance', 'label': 'Malmö, ISSN 0348-0909,44, 1976Även utg. i serien Malmö-litteratur, ISSN0348-0917', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Malmö, ISSN 0348-0909,44, 1976'}, 'extent': [{'@type': 'Extent', 'label': 's. 98-124'}]}], 'hasNote': [{'@type': 'Note', 'label': 'Fullständig beskrivning (OCR) ur SHBD: Andersson, Per, Malmö-litteratur : bibliografiska noteringar för år1975 : (med tillägg från föregående år). - I: Malmö, ISSN 0348-0909,44, 1976, s. 98-124. - Även utg. i 

In [213]:
extent = [{"@id": i["@id"], "extent": i["extent"][0]["label"]} for i in instances if "extent" in i and i["extent"][0]["label"][0].isalpha()]
extent_df = pd.json_normalize(extent)
extent_df.info()
extent_df.head(2)

pd.DataFrame(extent_df.value_counts(subset=["extent"])).head(50)


<class 'pandas.DataFrame'>
RangeIndex: 27983 entries, 0 to 27982
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   @id     27983 non-null  str  
 1   extent  27983 non-null  str  
dtypes: str(2)
memory usage: 437.4 KB


,count
extent,
s. 4,1004
s. 2,410
s. 16,79
s. 12,76
s. 1,74
s.4,68
bl. 2,46
s. 3,44
s.2,35


### Inspect seriesStatement

In [214]:
series_membership = [{"@id": i["@id"], "seriesMembership": i["seriesMembership"][0]} for i in instances if "seriesMembership" in i]

print(series_membership[:3])    

[{'@id': 'https://libris-qa.kb.se/dataset/shb/10#it', 'seriesMembership': {'@type': 'Instance', 'label': 'Specialarbete / Bibliotekshögskolan, ISSN 0347-1128 ; 1976:158', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Specialarbete / Bibliotekshögskolan, ISSN 0347-1128 ; 1976:158'}, 'identifiedBy': {'@type': 'ISSN', 'value': '0347-1128'}}}, {'@id': 'https://libris-qa.kb.se/dataset/shb/12#it', 'seriesMembership': {'@type': 'Instance', 'label': 'Sörmländska handlingar, ISSN0346-8097 ; 35', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Sörmländska handlingar, ISSN0346-8097 ; 35'}, 'identifiedBy': {'@type': 'ISSN', 'value': '0346-8097'}}}, {'@id': 'https://libris-qa.kb.se/dataset/shb/28#it', 'seriesMembership': {'@type': 'Instance', 'label': 'Acta Bibliothecae regiae Stockholmiensis,ISSN 0065-1060 ; 28', 'hasTitle': {'@type': 'Title', 'mainTitle': 'Acta Bibliothecae regiae Stockholmiensis,ISSN 0065-1060 ; 28'}, 'identifiedBy': {'@type': 'ISSN', 'value': '0065-1060'}}}]


In [215]:
series_df = pd.json_normalize(series_membership)
series_df.info()
series_df.head(30)

<class 'pandas.DataFrame'>
RangeIndex: 5404 entries, 0 to 5403
Data columns (total 9 columns):
 #   Column                                    Non-Null Count  Dtype 
---  ------                                    --------------  ----- 
 0   @id                                       5404 non-null   str   
 1   seriesMembership.@type                    5404 non-null   str   
 2   seriesMembership.label                    5404 non-null   str   
 3   seriesMembership.hasTitle.@type           5290 non-null   str   
 4   seriesMembership.hasTitle.mainTitle       5290 non-null   str   
 5   seriesMembership.identifiedBy.@type       101 non-null    str   
 6   seriesMembership.identifiedBy.value       101 non-null    str   
 7   seriesMembership.responsibilityStatement  136 non-null    str   
 8   seriesMembership.extent                   58 non-null     object
dtypes: object(1), str(8)
memory usage: 380.1+ KB


,@id,seriesMembership.@type,seriesMembership.label,seriesMembership.hasTitle.@type,seriesMembership.hasTitle.mainTitle,seriesMembership.identifiedBy.@type,seriesMembership.identifiedBy.value,seriesMembership.responsibilityStatement,seriesMembership.extent
0,https://libris-qa.kb.se/dataset/shb/10#it,Instance,"Specialarbete / Bibliotekshögskolan, ISSN 0347...",Title,"Specialarbete / Bibliotekshögskolan, ISSN 0347...",ISSN,0347-1128,NaN,NaN
1,https://libris-qa.kb.se/dataset/shb/12#it,Instance,"Sörmländska handlingar, ISSN0346-8097 ; 35",Title,"Sörmländska handlingar, ISSN0346-8097 ; 35",ISSN,0346-8097,NaN,NaN
2,https://libris-qa.kb.se/dataset/shb/28#it,Instance,"Acta Bibliothecae regiae Stockholmiensis,ISSN ...",Title,"Acta Bibliothecae regiae Stockholmiensis,ISSN ...",ISSN,0065-1060,NaN,NaN
3,https://libris-qa.kb.se/dataset/shb/38#it,Instance,Bibliografiska skrifter /Föreningen Malmö stad...,Title,Bibliografiska skrifter /Föreningen Malmö stad...,ISSN,0346-6884,NaN,NaN
4,https://libris-qa.kb.se/dataset/shb/52#it,Instance,"Småskriftserien / Föreningen Gamla Vadstena, I...",Title,"Småskriftserien / Föreningen Gamla Vadstena, I...",ISSN,0426-6587,NaN,NaN
5,https://libris-qa.kb.se/dataset/shb/58#it,Instance,"Skrifter / utgivna av svenskaRiksarkivet, ISSN...",Title,"Skrifter / utgivna av svenskaRiksarkivet, ISSN...",ISSN,0346-8488,NaN,NaN
6,https://libris-qa.kb.se/dataset/shb/74#it,Instance,"Dalarnas museumsserie av småskrifter, ISSN 034...",Title,"Dalarnas museumsserie av småskrifter, ISSN 034...",ISSN,0346-6949,NaN,NaN
7,https://libris-qa.kb.se/dataset/shb/78#it,Instance,Skriftserie / utgiven av Kulturnämnden i Kungä...,Title,Skriftserie / utgiven av Kulturnämnden i Kungä...,ISSN,0347-9072,NaN,NaN
8,https://libris-qa.kb.se/dataset/shb/81#it,Instance,Heddelande från Historiska institutionen i Lun...,Title,Heddelande från Historiska institutionen i Lun...,ISSN,0346-8631,NaN,NaN
9,https://libris-qa.kb.se/dataset/shb/82#it,Instance,Delrapport inom UHÅ-projektet Forskarutbildnin...,Title,Delrapport inom UHÅ-projektet Forskarutbildnin...,NaN,NaN,NaN,NaN


In [216]:
most_common_titles = pd.DataFrame(series_df.value_counts(subset=["seriesMembership.hasTitle.mainTitle"]))

most_common_titles.head(20)

,count
seriesMembership.hasTitle.mainTitle,
GHT 6/4 1957,7
Särtr. ur Vasabladet,6
Skolminnen,5
Festskr. utg. av Teol. fak. i Upps. 1941,5
tr. Sthlm,4
Hembygdsböckerna,4
UNT 1956: julnr,4
UNT 1957: julnr,4
Värt att veta,3


In [217]:
series_df[series_df["seriesMembership.hasTitle.mainTitle"] == "UNT 1959: julnr"]

,@id,seriesMembership.@type,seriesMembership.label,seriesMembership.hasTitle.@type,seriesMembership.hasTitle.mainTitle,seriesMembership.identifiedBy.@type,seriesMembership.identifiedBy.value,seriesMembership.responsibilityStatement,seriesMembership.extent
3555,https://libris-qa.kb.se/dataset/shb/44896#it,Instance,UNT 1959: julnr,Title,UNT 1959: julnr,NaN,NaN,NaN,NaN
4386,https://libris-qa.kb.se/dataset/shb/51696#it,Instance,UNT 1959: julnr,Title,UNT 1959: julnr,NaN,NaN,NaN,NaN
4387,https://libris-qa.kb.se/dataset/shb/51705#it,Instance,UNT 1959: julnr,Title,UNT 1959: julnr,NaN,NaN,NaN,NaN


### Count properties and subjects

#### Properties

In [218]:
for key, count in property_counts.most_common():
    print(f"{count:>5}  {key}")

79114  @id
79114  @type
79114  category
79114  instanceOf
79114  instanceOf.@type
79114  instanceOf.category
78970  hasTitle
78970  hasTitle.@type
78970  hasTitle.mainTitle
78970  hasNote
78376  instanceOf.subject
67759  responsibilityStatement
45968  extent
26566  partOf
 5404  seriesMembership
 3831  hasTitle.subtitle


#### Subjects

In [219]:
for key, count in subject_counts.most_common():

    print(f"{count:>5}  {key}")

46862  https://id.kb.se/term/sao/Sverige
20582  https://id.kb.se/term/sao/Historia
10501  https://id.kb.se/term/sao/Biografier
 4155  https://id.kb.se/term/sao/Kulturhistoria
 3840  https://id.kb.se/term/sao/Kyrkohistoria
 3651  https://id.kb.se/term/sao/1611-1718%20%28stormaktstiden%2C%20Sverige%29
 3481  https://id.kb.se/term/sao/1654-1718%20%28karolinska%20tiden%2C%20Sverige%29
 2921  https://id.kb.se/term/sao/Finland
 2813  https://id.kb.se/term/sao/Lokalhistoria
 2804  https://id.kb.se/term/sao/Ekonomisk%20historia
 2279  https://id.kb.se/term/sao/Utbildning--historia
 2234  https://id.kb.se/term/sao/1800-talet
 2215  https://id.kb.se/term/sao/Sverige--Sk%C3%A5ne
 2162  https://id.kb.se/term/sao/Milit%C3%A4rhistoria
 2042  https://id.kb.se/term/sao/R%C3%A4ttshistoria
 1813  https://id.kb.se/term/sao/1772-1809%20%28gustavianska%20tiden%2C%20Sverige%29
 1748  https://id.kb.se/term/sao/Forntiden
 1680  https://id.kb.se/term/sao/Sverige--Sm%C3%A5land
 1542  https://id.kb.se/term/sao/1

# Random stuff

In [ ]:
headers = {"Accept": "application/ld+json"}

query_string = f"instanceType:PhysicalResource title:({shbd_prepepd['mainTitle']}) title:({shbd_prepepd['subtitle']}) contributor:{shbd_prepepd['responsibility_statement']}* {shbd_prepepd['part_of_issn']} {shbd_prepepd['issn_from_note']}"


params = {"_q": "title:Hembergska+huset+i+Simrishamn contributor:Ehrnberg, G.*",
          #"_embellished": "false", Den här verkar inte göra något
          "_lens": "chips", # Den här behöver vara i plural
          "_stats": "false",
          "limit": 10}

res = requests.get("http://libris.kb.se/find?", params = params, headers=headers)
res.raise_for_status()
print(res.url)

print("Status:", res.status_code)
print("Number of results:", res.json()["totalItems"])
print("\nResult keys:", *res.json().keys(), sep=", ")

# Var finns den vanliga bibliografiska datan?
records = res.json()["items"]
print("\nItem keys:", *records[0].keys(), sep=", ")

print(records[0])


http://libris.kb.se/find?_q=title%3AHembergska%2Bhuset%2Bi%2BSimrishamn+contributor%3AEhrnberg%2C+G.%2A&_lens=chips&_stats=false&limit=10
Status: 200
Number of results: 1

Result keys:, @type, @id, search, itemOffset, itemsPerPage, totalItems, first, last, items, maxItems, @context

Item keys:, @type, meta, _categoryByCollection, @id, @reverse, hasTitle, language, contribution, reverseLinks
{'@type': 'Monograph', 'meta': {'mainEntity': {'@id': 'https://libris.kb.se/vc55wfk63mjdwll#work'}, '@type': 'VirtualRecord', '@id': 'https://libris.kb.se/vc55wfk63mjdwll#work-record'}, '_categoryByCollection': {'@none': [{'@type': 'ContentType', 'meta': {'mainEntity': {'@id': 'https://id.kb.se/term/rda/Text'}, '@type': 'Record', '@id': 'https://libris.kb.se/pnpsnkg0r5b6x092'}, '@id': 'https://id.kb.se/term/rda/Text', 'sameAs': [{'@id': 'https://id.kb.se/term/rda/content/text'}, {'@id': 'https://id.kb.se/term/rda/content/txt'}], 'prefLabelByLang': {'sv': 'Text', 'en': 'Text'}, 'code': 'txt', 'inSche

In [104]:
print(res.json()["stats"])

{'_predicates': [], 'sliceByDimension': {'librissearch:instanceType': {'dimension': 'librissearch:instanceType', 'observation': [{'totalItems': 1, 'view': {'@id': '/find?_q=title:Hembergska%2Bhuset%2Bi%2BSimrishamn+contributor:Ehrnberg%2C+G.*+instanceType:PhysicalResource'}, 'object': {'@id': 'PhysicalResource', '@type': 'Class', 'subClassOf': [{'@id': 'https://id.kb.se/vocab/Instance'}, {'@id': 'http://purl.org/dc/terms/PhysicalResource'}, {'@type': 'Restriction', 'onProperty': {'@id': 'https://id.kb.se/vocab/category'}, 'owl:onClass': {'hasValue': {'@id': 'https://id.kb.se/term/saobf/PhysicalForm'}, 'onProperty': {'@id': 'https://id.kb.se/vocab/broaderTransitive'}}, 'owl:minQualifiedCardinality': 1}], 'isDefinedBy': {'@id': 'https://id.kb.se/vocab/'}, 'labelByLang': {'en': 'Physical resource', 'sv': 'Fysisk resurs'}}}], 'maxItems': 100, '_connective': 'AND'}, 'librissearch:instanceCategory': {'dimension': 'librissearch:instanceCategory', 'observation': [{'totalItems': 1, 'view': {'@i

In [50]:
import re
rest = "Swedenborg : sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej"
#rest = "Egerbladh, Ossian, Ur Lappmarkens bebyggelsehistoria. Umeå. 1-8. Se SHB 1961/70:7809.9 : Barsele : minnesskrift med anledning av byns tvåhundraåriga tillvaro.1970. 98 s.10 : Stensele 1741-1860 : de hundra äldsta nybyggesupptagningarna.1972. 237 s.11 : Fyra gamla Lyckselebyar : Björksele, Brattfors, Falträsk, Vägsele :denna utredning har utförts med anledning av Lycksele sockens 300-årsjubileum. 1973. 94 s. : ill."
subtitle = ""
title= ""
if ' : ' in rest:
	title, rest = rest.split(' : ', 1)
	print (title)
	print(rest)

	if ':' in rest:
		parts =  re.split(r" ([./])", rest, maxsplit=1)
		subtitle = parts[0]
		if len(parts) > 1:
			print(parts)
			rest = "".join(parts[1:])
print()
print(title)
print(subtitle)
print(rest)



Swedenborg
sökaren i naturens och andens värld :hans verk och efterföljd / Carl / Hej
['sökaren i naturens och andens värld :hans verk och efterföljd', '/', ' Carl / Hej']

Swedenborg
sökaren i naturens och andens värld :hans verk och efterföljd
/ Carl / Hej


In [ ]:
csv_file = f"{home}/code/libris/repositories/librisxl/whelktool/scripts/dataimports/shb/data/mönster per materialtyp_rows.tsv"
patterns = pd.read_csv(csv_file, sep="\t", header=None, names=["year_range", "type", "pattern"])
patterns["pattern"] = patterns["pattern"].str.strip().str.replace(r"\s+", " ", regex=True)

patterns.info()
patterns.head(2)

<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   year_range  44 non-null     str  
 1   type        44 non-null     str  
 2   pattern     44 non-null     str  
dtypes: str(3)
memory usage: 1.3 KB


,year_range,type,pattern
0,1771-1874,Monografi,"Efternamn, Förnamn [initial]., Titel. undertit..."
1,1875-1900,Monografi,"Efternamn, Förnamn [initial]., Titel. undertit..."


In [265]:
SYNTAX_ERAS = {
    "1771-1874": "early", 
    "1875-1900": "early",
    "1901-1920": "early",
    "1921-1935": "parenthesized",  # Serietillhörighet och källpublikation anges inom parentes. Sidor anges efter Ort/år
    "1936-1950": "parenthesized",  # -||- . -||-
    "1951-1960": "parenthesized", # Serietillhörighet och källpublikation anges inom parentes. Sidor anges före Ort/år
    "1961-1970": "dash_style", # Serietillhörighet och källpublikation anges efter ". -". Sidor anges efter Ort/år
    "1971-1975": "isbd_transition", # Kolon ":" mellan huvudtitel och undertitel. Serietillhörighet anges inom parentes efter ". -". Bidrag: Källpublikation anges efter ". - I: "
    "1976": "isbd", # -||- ". - " anges före nytt avsnitt (utgivning, omfång, serietillhörighet). -||- . -||- . ISSN anges
}

patterns["syntax_era"] = patterns["year_range"].map(SYNTAX_ERAS)

In [267]:
unique_patterns = patterns.groupby(["pattern", "syntax_era"], as_index = False).agg({'year_range': ', '.join, 'type': 'first', 'syntax_era': 'first'})
unique_patterns["case"] = unique_patterns["year_range"] + " " + unique_patterns["type"]

unique_patterns.info()
unique_patterns.head(50)



<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   pattern     33 non-null     str  
 1   year_range  33 non-null     str  
 2   type        33 non-null     str  
 3   syntax_era  33 non-null     str  
 4   case        33 non-null     str  
dtypes: str(5)
memory usage: 1.4 KB


,pattern,year_range,type,syntax_era,case
0,"Efternamn, Förnamn [initial]., Titel. Publ-tit...",1771-1874,Bidrag (tidningsartikel),early,1771-1874 Bidrag (tidningsartikel)
1,"Efternamn, Förnamn [initial]., Titel. Publ-tit...",1875-1900,Bidrag (tidningsartikel),early,1875-1900 Bidrag (tidningsartikel)
2,"Efternamn, Förnamn [initial]., Titel. undertit...",1936-1950,Bidrag (tidningsartikel),parenthesized,1936-1950 Bidrag (tidningsartikel)
3,"Efternamn, Förnamn [initial]., Titel. undertit...",1936-1950,Bidrag,parenthesized,1936-1950 Bidrag
4,"Efternamn, Förnamn [initial]., Titel. undertit...",1936-1950,Monografi,parenthesized,1936-1950 Monografi
5,"Efternamn, Förnamn [initial]., Titel. undertit...",1901-1920,Bidrag (tidningsartikel),early,1901-1920 Bidrag (tidningsartikel)
6,"Efternamn, Förnamn [initial]., Titel. undertit...","1771-1874, 1875-1900, 1901-1920",Bidrag,early,"1771-1874, 1875-1900, 1901-1920 Bidrag"
7,"Efternamn, Förnamn [initial]., Titel. undertit...","1771-1874, 1875-1900, 1901-1920",Monografi,early,"1771-1874, 1875-1900, 1901-1920 Monografi"
8,"Efternamn, Förnamn, Titel : undertitel. - ""I:""...",1971-1975,Bidrag (tidningsartikel),isbd_transition,1971-1975 Bidrag (tidningsartikel)
9,"Efternamn, Förnamn, Titel : undertitel. - ""I:""...",1976,Bidrag (tidningsartikel),isbd,1976 Bidrag (tidningsartikel)


In [ ]:

test_cases = ""

for record in unique_patterns.to_dict(orient="records"):
    
    
    test_cases += f"""
def test_parse_note_{record["case"].replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace(":", "").replace(",", "")}():
	instance = {{"hasNote": [{{"label": "{record["pattern"]}"}}]}}
	
	result = parse_note(instance, "{record["syntax_era"]}")

	assert result["category"] == None
	assert result["hasTitle"]["mainTitle"] == None
	assert result["hasTitle"]["subtitle"] == None
	assert result["responsibilityStatement"] == None
	assert result["extent"] == None
	assert result["partOf"] == None
	assert result["seriesMembership"] == None
	assert result["hasNote"] == None
		"""

print(test_cases)